# QuackWeave stabilization with SQLLogicTest

**Use DuckDB’s native SQLLogicTest runner as the acceptance oracle for MVP2.** The `.test` files own the assertions. Fixture code builds a fresh Ladybug database, stages packages, and records facts that SQL cannot observe during a query.

The reference suite ran against QuackWeave MVP2 `cd3f54e`, Ladybug `v0.22.3`, and DuckDB `v2.0.1` at `8d78114e90a53823803660bf821f6073ac87da8c`.

| Profile | Native test files passed | SQLLogicTest blocks passed |
|---|---:|---:|
| MVP2 debug, relocated | 9 | 150 |
| MVP2 release, relocated | 9 | 150 |
| MVP1 Cypher-only | 1 | 4 |

Each profile also passed a separate runner version assertion. The reference suite covers 42 named case groups across the three profiles. Seven native-observation groups are specified for the stabilization exit gate. Keep the existing Rust safety tests while those observations are brought into SQLLogicTest.

[Fixture launcher](test/stabilization/run_sqllogictest.py) · [SQLLogicTest files](test/stabilization/sqllogic) · [MVP2 design](quack-weave-mvp2-technical-design.ipynb)

## DuckDB code and conventions

I inspected the registered `duckdb` project with `knowledge_context_pack_2`, `code_symbol_search`, and `code_read_symbol`. The worktree is newer than the product ABI pin. The indexed `result_helper.cpp` blob `c66a885c17abe4874a3dac480517d390f7007697` matches `v2.0.1`. The command and runner blobs differ, so I used the pinned source and tested the relevant conventions in the exact-version runner.

| Symbol or example | Design decision |
|---|---|
| `SQLLogicTestRunner::ExecuteFile` and `LoadDatabase` | Use a fresh DuckDB instance per file. Pass fixture paths with `test-env` and `--env-passthrough`. Use named connections when a cross-connection config change or recovery matters. [Runner source](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/test/sqlite/sqllogic_test_runner.cpp) |
| `Command::ExecuteQuery` and `TestResultHelper::CheckQueryResult` | Let the runner compare exact results. Use `rowsort` for row bags, `ORDER BY` for promised order, `typeof` or `DESCRIBE` for host types, and stable `QW_*` diagnostics for expected errors. [Command](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/test/sqlite/sqllogic_command.cpp) · [Result checks](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/test/sqlite/result_helper.cpp) |
| `RegisterSqllogictests` and `TryReadExactSQLLogicTestFilter` | Supply an exact absolute file list with `-f`. Check `--emit-test-events` terminal records and per-file assertion-block counts. [Registration](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/test/sqlite/test_sqllogictest.cpp) · [Unittest entry point](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/test/unittest.cpp) |
| Upstream extension load and global option tests | `LOAD` the packaged artifact and use the same DuckDB instance for cross-connection reset and prepared-read invalidation. [V2 load test](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/test/extension/load_extension_capi_v2.test) · [Global option test](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/test/extension/reset_global_extension_option.test) |

QuackWeave anchors are `GraphQuery::bind` for dialect and generation, `Scan::next` for the Arrow schema and cursor, and Ladybug `Session::execute` for native admission, prepare and dispatch. The existing acceptances and Rust tests provide the baseline case inventory. [DuckDB SQLLogicTest guide](https://duckdb.org/docs/stable/dev/sqllogictest/intro)

## Runner and fixture lifecycle

```mermaid
flowchart LR
    Build[Versioned artifact] --> Version[Verify DuckDB version]
    Version --> Seed[Seed isolated Ladybug graph]
    Seed --> Before[Observe initial graph]
    Before --> Run[Run exact SQLLogicTest manifest]
    Run --> After[Observe graph and package]
    After --> Audit[Assert facts in fixture_audit.test]
    Audit --> Evidence[Validate acceptance evidence]
```

- The launcher creates one scratch root per profile and uses [read_graph.cypher](test/stabilization/fixtures/read_graph.cypher) to provision duplicate edges, a self-loop, nullable properties, Unicode and punctuation, and 5,001 linked sample rows. It creates a corrupt graph and reserves a path for the denied export.
- The runner stages the extension under a path with spaces and removes loader environment overrides. The product artifact loads into the pinned DuckDB engine and uses the real Ladybug file.
- An independent Ladybug CLI session captures nodes, relationships and catalog names before and after the suite. SQL compares the canonical snapshots, including multiplicity. A second fixture captures footer fields, exported symbols, imports, digest pairs and export-file presence as CSV. SQL verifies those raw facts using the built-in CSV reader.
- The launcher checks nonzero exit, missing or duplicate terminal records, and per-file block counts. It includes the timeout case explicitly. Expected SQL values, diagnostics, schema and fixture verdicts stay in the `.test` files.

## Reference suite and coverage

| File | Case logic |
|---|---|
| [reads.test](test/stabilization/sqllogic/reads.test) | Fixed and variable-length Cypher, GQL/Cypher bag parity, case-insensitive dialect and alias, self-loops, nullable scalar values and types, all comparison operators, FILTER forms, Unicode/quotes, joins and empty results. |
| [bind.test](test/stabilization/sqllogic/bind.test) | Constant arguments, invalid and reset config, stale prepared reads, SQL schema arguments, GQL source diagnostics and closed subset, variable/alias scope, no fallback, and bind versus execute against a corrupt graph. |
| [admission.test](test/stabilization/sqllogic/admission.test) | Reject invalid native input, writes, batches, routines and export. Prove a lowered GQL query with an unknown property fails native preparation. Verify graph sentinel rows after rejects. |
| [scan.test](test/stabilization/sqllogic/scan.test) | Arrow name/type/order errors, nested output rejection, 5,001-row checksum, early close, connection recovery and batch quota reset. |
| [limits.test](test/stabilization/sqllogic/limits.test) | Both sides of existing 16 KiB, 256-token and 16-level input envelopes, signed integer bounds, and invalid Unicode scalars. |
| [host_access.test](test/stabilization/sqllogic/host_access.test) · [timeout.test_slow](test/stabilization/sqllogic/timeout.test_slow) | Host access recheck, native timeout and same-connection recovery. |
| [artifact.test](test/stabilization/sqllogic/artifact.test) · [feature_off.test](test/stabilization/sqllogic/feature_off.test) | The relocated package loads both dialects and no external GQL extension is loaded. The Cypher-only package reports `QW_GQL_UNAVAILABLE` for GQL. |
| [fixture_audit.test](test/stabilization/sqllogic/fixture_audit.test) | Independent graph and export checks, exact ABI footer, initializer, imports, and the three pinned GQL notices. |

The debug and release profiles each require 41 named groups. The feature-off profile requires one. The runner confirms all assertion blocks in each selected file completed.

## Native observation fixtures for stabilization exit

The public query result cannot expose native call counts, cursor lifetime or already-published client chunks. Capture each of those raw observations using test-only instrumentation. Assert the records in a future `observer_audit.test` while the packaged suite continues to verify SQL behavior.

| Case | Fixture | SQL oracle |
|---|---|---|
| `observer.parser` | `parser.csv` | AST output for scalar/escaped values and spans before and after Unicode; compare data to expected UTF-8 slices. |
| `observer.admission` | `events.csv` | AST rejection has zero prepare/execute; failed native prepare has one prepare and zero execute; read-only check precedes the single dispatch. |
| `observer.schema` | `events.csv` | One successful execute followed by rejected schema, zero pulls/publications, one release. |
| `observer.lifecycle` | `events.csv` | Per-invocation monotonic sequence, no reopen after error, bounded native pulls and release on every terminal path. |
| `observer.quota` | `batch_bytes.csv` | Test-derived byte threshold B; compare raw observed admission at B and rejection at B-1 without literal host-layout assumptions. |
| `observer.failure` | `faults.csv` | Deterministic test-only Arrow invalidity, timeout and execution failure plus observed stream chunks; acceptance of earlier complete batches only where specified. |
| `observer.policy` | `paths.csv` | Fixture barrier after bind, deterministic target swap, observed path mismatch and zero native open. |

Use records keyed by `fixture_version, case_id, invocation_id, sequence`. Capture the event and outcome at actual native calls or the DuckDB publication boundary. Include byte counts, diagnostics or path classification when a case calls for them. Validate case identity, unique and contiguous ordinals and explicit start/end records before counting events. Missing evidence must fail an expected zero-call case.

For successful GQL, assert `ast_admitted < prepare < prepared_read_only < execute < schema_accepted < first_publish < release` with exactly one execute and release. A failed prepare asserts one prepare, zero execute and zero publish. A schema rejection after execute asserts zero pulls and publications. Verify the observer tests by removing an observed release and inserting an early publication into scratch evidence.

Keep the actual timeout workload as a smoke test. The deterministic timeout observer also captures client-visible chunks. Treat a runner watchdog as a suite failure. Preserve the existing native fault-injection tests as evidence until the SQL observer parity check is complete.

[Native tests](crates/native/src/ladybug/tests.rs) · [Real C ABI observer](crates/native/src/ladybug/ffi/observed.rs) · [Core scan tests](crates/core/src/qw_query/tests.rs) · [GQL parser tests](crates/native/src/gql/tests.rs)

## Reproduce the pinned run

Use `unittest` built from DuckDB `v2.0.1`. The CLI binary and the Jute SQL kernel cannot run these test files. The local proof linked the pinned upstream runner, test helpers and RE2 to the release `libduckdb.dylib`. The launcher checked engine version and actual artifact loading.

```sh
cmake -S /path/to/duckdb-v2.0.1 -B /path/to/duckdb-build \
  -DCMAKE_BUILD_TYPE=Release -DBUILD_UNITTESTS=ON -DENABLE_UNITTEST_CPP_TESTS=OFF
cmake --build /path/to/duckdb-build --target unittest --parallel

make mvp2-deps
make mvp2 mvp2-release mvp1
python3 -B test/stabilization/run_sqllogictest.py \
  --duckdb-unittest /path/to/duckdb-build/test/unittest \
  --lbug /path/to/lbug-v0.22.3 \
  --extension build/mvp2/release/extension/quack_weave/quack_weave.duckdb_extension
```

Repeat the launcher with the MVP2 debug artifact. Use `build/debug/extension/quack_weave/quack_weave.duckdb_extension --without-gql` for the Cypher-only profile. Keep the file manifests and terminal JSON in CI. [DuckDB test build](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/test/CMakeLists.txt)

## Release acceptance gate

A complete mapping, successful SQL execution, successful fixture audit and complete native-observer evidence are required for acceptance. The diagram checks this decision logic. Runtime test reports and fixtures supply the inputs.

In [ ]:
flowchart TD
    SPEC["`@spec QW-STABILIZATION-ACCEPTANCE
@type Status = enum[unmapped, sql_failed_or_missing, fixture_failed_or_missing, observer_pending, accepted]
@input mappings_complete: Bool
@input sql_evidence_complete: Bool
@input fixture_evidence_complete: Bool
@input observer_evidence_complete: Bool
@output status: Status
@output release_accepted: Bool
@requires DOMAIN: true`"]
    A["`@branch UNMAPPED
@when not mappings_complete
@ensures STATUS_A: status = unmapped
@ensures GATE_A: release_accepted = false`"]
    B["`@branch SQL_GAP
@when mappings_complete and not sql_evidence_complete
@ensures STATUS_B: status = sql_failed_or_missing
@ensures GATE_B: release_accepted = false`"]
    C["`@branch FIXTURE_GAP
@when mappings_complete and sql_evidence_complete and not fixture_evidence_complete
@ensures STATUS_C: status = fixture_failed_or_missing
@ensures GATE_C: release_accepted = false`"]
    D["`@branch OBSERVER_GAP
@when mappings_complete and sql_evidence_complete and fixture_evidence_complete and not observer_evidence_complete
@ensures STATUS_D: status = observer_pending
@ensures GATE_D: release_accepted = false`"]
    E["`@branch ACCEPTED
@when mappings_complete and sql_evidence_complete and fixture_evidence_complete and observer_evidence_complete
@ensures STATUS_E: status = accepted
@ensures GATE_E: release_accepted = true`"]
    VERIFY["`@verify CONSISTENT: witness consistency
@verify DETERMINISTIC: prove determinism
@verify COVERED: prove partition_coverage
@verify EXCLUSIVE: prove partition_exclusive
@verify EACH_STATUS: witness each status`"]
    SPEC --> A --> VERIFY
    SPEC --> B --> VERIFY
    SPEC --> C --> VERIFY
    SPEC --> D --> VERIFY
    SPEC --> E --> VERIFY

## Stabilization implementation sequence

| Step | Deliverable and exit criterion |
|---|---|
| S1 · SQL baseline | Run the reference suite on pinned debug, release and Cypher-only packages. Assert graph and package observations through SQL. Completed locally. |
| S2 · Observer evidence | Capture the seven native observation groups. Add fixture schema validation, SQL oracles and negative controls for bad sequences, missing events and malformed fault evidence. |
| S3 · CI entry point | Provision and pin the native runner. Run the full profile matrix and check exact expected block counts. Save runtime records and prove required evidence membership. |
| S4 · Consolidation | Compare old versus new evidence case by case before retiring redundant behavioral assertions. Keep useful low-level fault injection and sanitizer checks. |

The stabilization exit requires the entire declared case/profile inventory. Expected failures must reach the intended QuackWeave diagnostic or observer predicate. Bootstrap failure, skipped tests and missing files cannot satisfy a negative test.

## JEV Compile and Z3 coverage evidence

Coverage uses the catalog's `data_integrity.foreign_key` rule in verification mode. Each required pair of profile and exact case set must exist in the evidence relation. Actual plan evidence came from case markers in the suite and the observer design. Actual runtime evidence came from terminal test records with exact block counts.

| Check | JEV gate and weakest confidence | Z3 | Persisted result |
|---|---|---|---|
| Required suite and observer plan | open · 0.97 | `sat` | `sol_eaae3c8b47e94e3d` |
| Release graph audit removed from the plan | open · 0.98 | `unsat` | `sol_e644b244e0e849c6` |
| Observed runtime for the shipped reference | open · 0.97 | `sat` | `sol_01f420cc14674b0b` |
| Observed runtime against the full stabilization gate | open · 0.88 | `unsat` | `sol_e29286db658c40a1` |

JEV version `jev-1.14.0` proposed one binding per check. The executed rule, mode, subjects and normalized facts matched the candidates. Each solver receipt was reloaded. The first model had 90 required case/profile pairs. The current runtime satisfies all 83 reference-suite pairs. The seven native observer pairs still require evidence.

### Replay the finite set check

The catalog caps solver variables, so the case lists are encoded as exact integer masks per profile. Case number `n` is bit `n - 1`. Each profile has one active row and every case ID is unique within it. The mask is injective over the 49 declared case IDs. Bit masks preserve the entire membership set.

| Profile | Profile code |
|---|---:|
| Debug | 1 |
| Release | 2 |
| Cypher-only | 3 |
| Native observer | 4 |

Use `solve_rule_spec` with `data_integrity.foreign_key`. Encode each table below as one row per profile in the `required_evidence` or `case_evidence` relation, with integer columns `profile` and `case_mask`. Set both cells present. Use the two columns in that order in the `coverage` foreign key from required to evidence. Send one subject `coverage` with empty parameters/unknowns to Jev. Execute the vetted candidate with `solve_rules`, persistence enabled.

```json
{
  "plan": {
    "required": {
      "debug": 4397979402239,
      "release": 4397979402239,
      "mvp1": 67108864,
      "observed": 558551906910208
    },
    "evidence": {
      "debug": 4397979402239,
      "release": 4397979402239,
      "mvp1": 67108864,
      "observed": 558551906910208
    },
    "expected_status": "sat",
    "receipt": "sol_eaae3c8b47e94e3d"
  },
  "missing": {
    "required": {
      "debug": 4397979402239,
      "release": 4397979402239,
      "mvp1": 67108864,
      "observed": 558551906910208
    },
    "evidence": {
      "debug": 4397979402239,
      "release": 4397710966783,
      "mvp1": 67108864,
      "observed": 558551906910208
    },
    "expected_status": "unsat",
    "receipt": "sol_e644b244e0e849c6"
  },
  "runtime": {
    "required": {
      "debug": 4397979402239,
      "release": 4397979402239,
      "mvp1": 67108864
    },
    "evidence": {
      "debug": 4397979402239,
      "release": 4397979402239,
      "mvp1": 67108864
    },
    "expected_status": "sat",
    "receipt": "sol_01f420cc14674b0b"
  },
  "gap": {
    "required": {
      "debug": 4397979402239,
      "release": 4397979402239,
      "mvp1": 67108864,
      "observed": 558551906910208
    },
    "evidence": {
      "debug": 4397979402239,
      "release": 4397979402239,
      "mvp1": 67108864
    },
    "expected_status": "unsat",
    "receipt": "sol_e29286db658c40a1"
  }
}
```

## Case identity and traceability

| Case number | Required case | Assertion source |
|---:|---|---|
| 1 | `read.setup` | [reads.test](test/stabilization/sqllogic/reads.test#L9) |
| 2 | `read.cypher` | [reads.test](test/stabilization/sqllogic/reads.test#L16) |
| 3 | `read.gql` | [reads.test](test/stabilization/sqllogic/reads.test#L30) |
| 4 | `read.parity` | [reads.test](test/stabilization/sqllogic/reads.test#L37) |
| 5 | `read.filter` | [reads.test](test/stabilization/sqllogic/reads.test#L50) |
| 6 | `read.self` | [reads.test](test/stabilization/sqllogic/reads.test#L122) |
| 7 | `read.scalar` | [reads.test](test/stabilization/sqllogic/reads.test#L128) |
| 8 | `read.join` | [reads.test](test/stabilization/sqllogic/reads.test#L135) |
| 9 | `read.empty` | [reads.test](test/stabilization/sqllogic/reads.test#L142) |
| 10 | `read.literal` | [reads.test](test/stabilization/sqllogic/reads.test#L153) |
| 11 | `bind.config` | [bind.test](test/stabilization/sqllogic/bind.test#L13) |
| 12 | `bind.generation` | [bind.test](test/stabilization/sqllogic/bind.test#L112) |
| 13 | `bind.parse` | [bind.test](test/stabilization/sqllogic/bind.test#L127) |
| 14 | `bind.subset` | [bind.test](test/stabilization/sqllogic/bind.test#L139) |
| 15 | `bind.scope` | [bind.test](test/stabilization/sqllogic/bind.test#L221) |
| 16 | `bind.plan` | [bind.test](test/stabilization/sqllogic/bind.test#L242) |
| 17 | `admission.setup` | [admission.test](test/stabilization/sqllogic/admission.test#L11) |
| 18 | `admission.cypher` | [admission.test](test/stabilization/sqllogic/admission.test#L18) |
| 19 | `admission.prepare` | [admission.test](test/stabilization/sqllogic/admission.test#L88) |
| 20 | `admission.guard` | [admission.test](test/stabilization/sqllogic/admission.test#L94) |
| 21 | `scan.setup` | [scan.test](test/stabilization/sqllogic/scan.test#L9) |
| 22 | `scan.schema` | [scan.test](test/stabilization/sqllogic/scan.test#L16) |
| 23 | `scan.multibatch` | [scan.test](test/stabilization/sqllogic/scan.test#L37) |
| 24 | `scan.budget` | [scan.test](test/stabilization/sqllogic/scan.test#L54) |
| 25 | `host.disable` | [host_access.test](test/stabilization/sqllogic/host_access.test#L9) |
| 26 | `scan.timeout` | [timeout.test_slow](test/stabilization/sqllogic/timeout.test_slow#L9) |
| 27 | `dialect.off` | [feature_off.test](test/stabilization/sqllogic/feature_off.test#L9) |
| 28 | `artifact.load` | [artifact.test](test/stabilization/sqllogic/artifact.test#L9) |
| 29 | `fixture.graph` | [fixture_audit.test](test/stabilization/sqllogic/fixture_audit.test#L15) |
| 30 | `fixture.export` | [fixture_audit.test](test/stabilization/sqllogic/fixture_audit.test#L22) |
| 31 | `fixture.footer` | [fixture_audit.test](test/stabilization/sqllogic/fixture_audit.test#L28) |
| 32 | `fixture.exports` | [fixture_audit.test](test/stabilization/sqllogic/fixture_audit.test#L36) |
| 33 | `fixture.notices` | [fixture_audit.test](test/stabilization/sqllogic/fixture_audit.test#L53) |
| 34 | `limits.bytes` | [limits.test](test/stabilization/sqllogic/limits.test#L15) |
| 35 | `limits.tokens` | [limits.test](test/stabilization/sqllogic/limits.test#L24) |
| 36 | `limits.depth` | [limits.test](test/stabilization/sqllogic/limits.test#L33) |
| 37 | `limits.literal` | [limits.test](test/stabilization/sqllogic/limits.test#L44) |
| 38 | `bind.config_reject` | [bind.test](test/stabilization/sqllogic/bind.test#L66) |
| 39 | `read.comparison` | [reads.test](test/stabilization/sqllogic/reads.test#L95) |
| 40 | `bind.shape` | [bind.test](test/stabilization/sqllogic/bind.test#L170) |
| 41 | `admission.empty_controls` | [admission.test](test/stabilization/sqllogic/admission.test#L77) |
| 42 | `fixture.setup` | [fixture_audit.test](test/stabilization/sqllogic/fixture_audit.test#L11) |
| 43 | `observer.parser` | Planned native observer fixture |
| 44 | `observer.admission` | Planned native observer fixture |
| 45 | `observer.schema` | Planned native observer fixture |
| 46 | `observer.lifecycle` | Planned native observer fixture |
| 47 | `observer.quota` | Planned native observer fixture |
| 48 | `observer.failure` | Planned native observer fixture |
| 49 | `observer.policy` | Planned native observer fixture |

An expected case maps to a validated file and profile, and a file maps to a terminal runner record with all blocks passing. The observer cases map to future fixture results with required start/end records. The model validates this mapping and the missing-evidence controls. SQLLogicTest and the observation fixtures provide behavioral evidence.

## Runtime negative controls

The exact-version runner rejected the fixture audit after a scratch after-snapshot was changed, with the failure attributed to the SQL equality assertion. It also rejected the Cypher-only profile when given the MVP2 artifact. These controls exercised the actual test harness and complement the JEV missing-evidence checks.

| Evidence | Local output |
|---|---|
| MVP2 debug | [Terminal records](build/stabilization-debug.log) |
| MVP2 release | [Terminal records](build/stabilization-release.log) |
| MVP1 feature-off | [Terminal records](build/stabilization-feature_off.log) |
| Feature-off case given the wrong artifact | [Expected failure](build/stabilization-negative-feature.log) |